# Variant 1 — TaskTransformer (complete) + TimeTransformer
Pipeline: token combinato (regione+task) → predice il prossimo token, poi il tempo.

In [ ]:
import os
import torch
import pandas as pd
from config import DATA_DIR

from pm4py.algo.conformance.alignments.petri_net import algorithm as alignments
from pm4py.objects.log.obj import Trace, Event, EventLog

from core import TaskTransformer, TimeTransformer
from core.training import train_task, train_time
from utils import get_decoding, get_encoding, hamming_distance, edit_distance_weighted_levenshtein

device = 'cuda' if torch.cuda.is_available() else 'cpu'
#print(device)
DATA_FILE = DATA_DIR / 'prepared_data.pt'

In [ ]:
info = torch.load(DATA_FILE, map_location=device, weights_only=False)
n = info['n']

data_task = {
    'train_complete': info['data_complete'][:n],
    'val_complete': info['data_complete'][n:],
}
data_time = {
    'train_complete': info['data_complete'][:n],
    'val_complete': info['data_complete'][n:],
    'train_times': info['data_times'][:n],
    'val_times': info['data_times'][n:],
}

vocab_size = info['vocab_size_complete']
num_regions = info['num_regions']
num_tasks = info['num_tasks']

decode_complete = lambda b: [info['id_to_bit_complete'][x] for x in b]
encode_complete = lambda a: [info['bit_to_id_complete'][tuple(x)] for x in a]

net = info['net']

#print(f'vocab={vocab_size}, n_train={n}')

In [ ]:
param_task_region_transformer_default = dict(block_size=256, n_embd=256, n_head=8, n_layer=3, dropout=0.30, lr=3e-4, weight_decay=0.01, batch_size=16, max_iters=2500, eval_iters=200, eval_interval=100, patience=4, use_swa=True, swa_start_ratio=0.6, diverge_threshold=1.5)
param_time_transformer_default = dict(block_size=64,  n_embd=128, n_head=8, n_layer=3, dropout=0.30, lr=3e-4, weight_decay=0.01, batch_size=16, max_iters=2500, eval_iters=200, eval_interval=100, patience=4, use_swa=True, swa_start_ratio=0.6, diverge_threshold=1.5)

params = torch.load(DATA_DIR / 'v1_best_params.pt', map_location=device, weights_only=False) if os.path.exists(DATA_DIR / 'v1_best_params.pt') else None

p_task_region = param_task_region_transformer_default
p_time = param_time_transformer_default
if params is not None:
    p_task_region = {**params['TaskTransformer'], 'max_iters': 10000, 'eval_iters': 100, 'eval_interval': 100, 'patience': 6, 'use_swa': True, 'swa_start_ratio': 0.6, 'diverge_threshold': 1.5}
    p_time = {**params['TimeTransformer'],  'max_iters': 10000, 'eval_iters': 100, 'eval_interval': 100, 'patience': 6, 'use_swa': True, 'swa_start_ratio': 0.6, 'diverge_threshold': 1.5}

print('Task Region params:', p_task_region)
print('Time params:', p_time)

In [ ]:
model_task = TaskTransformer(
    task_vocab_size=vocab_size,
    block_size=p_task_region['block_size'],
    n_embd=p_task_region['n_embd'],
    dropout=p_task_region['dropout'],
    n_head=p_task_region['n_head'],
    n_layer=p_task_region['n_layer'],
).to(device)

best_val_task = train_task(model_task, data_task, p_task_region, device, data_key='complete', printing=False)
print(f'TaskTransformer trained. Best val loss: {best_val_task:.4f}')

In [ ]:
model_time = TimeTransformer(
    vocab_size_task=vocab_size,
    block_size=p_time['block_size'],
    n_embd=p_time['n_embd'],
    dropout=p_time['dropout'],
    n_head=p_time['n_head'],
    n_layer=p_time['n_layer'],
    separated_regions=False,
).to(device)

best_val_time = train_time(model_time, data_time, p_time, device, printing=False)
print(f'TimeTransformer trained. Best val loss: {best_val_time:.4f}')

In [ ]:
max_new_tokens = 10000

sep_id = info['bit_to_id_complete'][tuple([0]*(num_regions+num_tasks))]
mean_sep_delta = info['data_times'][:n][info['data_complete'][:n] == sep_id].float().mean().item()

context = torch.tensor(encode_complete([[0]*(num_regions+num_tasks)]), dtype=torch.long, device=device).unsqueeze(0)
context_time = torch.tensor([mean_sep_delta], dtype=torch.float32, device=device).unsqueeze(0)

generated_ids = []
generated_times = []

for _ in range(max_new_tokens):
    next_id = model_task.predict_next_task(idx_task=context, block_size=p_task_region['block_size'])
    context = torch.cat((context, next_id), dim=1)
    context_aligned = context[:, 1:]   # allineamento offset

    next_time = model_time.predict_next_time(idx_tasks=context_aligned, idx_times=context_time, block_size=p_time['block_size'])
    context_time = torch.cat((context_time, next_time), dim=1)

    generated_ids.append(next_id.item())
    generated_times.append(next_time.item())

decoded = decode_complete(generated_ids)
'''for i, (step, t) in enumerate(zip(decoded, generated_times)):
    print(f'{i:02d}: {[int(b) for b in step]} — time={round(t,4)}')'''

In [ ]:
# Raggruppa la sequenza in tracce separate (separatore = vettore zero)
traces_generated, current = [], []
for step in decoded:
    bits = [int(b) for b in step]
    current.append(bits)
    if bits == [0]*(num_regions+num_tasks):
        if len(current) > 1:
            traces_generated.append(current)
        current = []

'''for i,trace in enumerate(traces_generated):
    print(f"{i}: {trace}")'''

In [ ]:
MIN_CONTEXT_STEPS = 100

cumulative = 0
skip_n = 1
for trace in traces_generated:
    if cumulative >= MIN_CONTEXT_STEPS:
        break
    cumulative += len(trace)
    skip_n += 1

#print(f"Warm-up: {skip_n} tracce saltate ({cumulative} step) | valutazione su {len(traces_generated)-skip_n} tracce")

In [ ]:
traces_decoded = get_decoding(traces_generated, net.regions, net.tasks)
#print(traces_decoded)

In [ ]:
classifier_dict_tasks = info['classifier_dict_tasks']
dict_task_step_encoding = info['dict_task_step_encoding']
current_trace_context = []
#traces_decoded_list = [step for trace in traces_decoded for step in trace]

tasks_previous = [0] * num_tasks
current_trace = 0
mae_values = []

for i, (step, t) in enumerate(zip(decoded, generated_times)):
    bits = [int(b) for b in step]
    is_sep = bits == [0]*(num_regions+num_tasks)
    tasks_step = bits[num_regions:]

    step_events = []
    for j, task in enumerate(tasks_step):
        if task != tasks_previous[j]:
            step_events.append(("start_" if task == 1 else "end_") + net.tasks[j])
    tasks_previous = tasks_step

    if len(step_events) == 1:
        event_name = step_events[0]
        if event_name in classifier_dict_tasks:
            rt, max_len = classifier_dict_tasks[event_name]
            ctx = list(reversed(current_trace_context))[:max_len]
            padded = ctx + ['PAD'] * (max_len - len(ctx))
            encoded = [dict_task_step_encoding.get(s, dict_task_step_encoding['PAD']) for s in padded]
            expected = round(rt.predict([encoded])[0], 4)
        else: # Non ci dovrebbe mai entrare in teoria
            expected = 0.0 if not current_trace_context else "n/d"
        if current_trace >= skip_n:
            mae_values.append(abs(t - expected))
        current_trace_context.append(event_name)
        note = ""
    elif len(step_events) == 0: # step che non genera eventi (es. cambia solo la regione)
        expected, note = "—", "(step senza evento)"
    else: # step malformato: accende/spegne 2 task insieme
        expected, note = "—", f"(step ambiguo: {step_events})"

    if is_sep:
        current_trace_context = []
        current_trace += 1

    #print(f'{i:02d}: {bits} — time={round(t,4)} | expected={expected} {note}')

mae = sum(mae_values) / len(mae_values) if mae_values else float('nan')
print(f'\nTime MAE (ignorando tracce warm-up, {len(mae_values)} step): {mae:.4f}')

In [ ]:
# Allineamento con conformance checking pm4py
check = ["P", "X", "L"]
start = tuple(["start_" + c for c in check])
end = tuple(["end_" + c for c in check if c!="L"]) # si escludono gli end loop
loop = tuple(["back_L"]) + tuple(["end_L"])
silent_prefixes = start + end + loop

# Creo i parametri per l'allineamento
model_cost, sync_cost = {}, {}
for t in net.net.transitions: # Prendo tutte le transizioni
    if t.label is None or (t.label is not None and t.label.startswith(silent_prefixes)): # Se è una transizione silente
        model_cost[t] = 0
        sync_cost[t] = 10000
    else: # Se è un task vero e proprio
        model_cost[t] = 10000
        sync_cost[t] = 0

alignment_params = {
    alignments.Parameters.PARAM_MODEL_COST_FUNCTION: model_cost,
    alignments.Parameters.PARAM_SYNC_COST_FUNCTION: sync_cost,
}

# Creiamo l'EventLog di ogni traccia per poi poterla allineare
eventlog_traces = EventLog()
for trace in traces_decoded:
    t = Trace()
    for activity in trace:
        t.append(Event({'concept:name': activity}))
    eventlog_traces.append(t)

aligned_traces = alignments.apply(eventlog_traces, net.net, net.initial_marking, net.final_marking, parameters=alignment_params)
'''for i,trace in enumerate(aligned_traces):
    print(f"{i}: {trace}")'''

In [ ]:
'''Codifichiamo le tracce allineate (per poi poterle confrontare con quelle generate dal transformer)'''

silent_prefixes = start + end + tuple(["back_L"])

aligned_traceEncoded_regions, aligned_traceEncoded_tasks = get_encoding(
    [[step for _, step in a['alignment'] if step and not step.startswith(silent_prefixes) and step != '>>']
     for a in aligned_traces],
    net.regions, net.tasks, net.open_clauses, net.end_clauses
)

#print(aligned_traceEncoded_regions)

df_aligned_traces = pd.concat([aligned_traceEncoded_regions, aligned_traceEncoded_tasks], axis=0)

In [ ]:
aligned_traces_encoded = []
aligned_trace_encoded = []
for element in df_aligned_traces.T.values:
    element = element.tolist()
    aligned_trace_encoded.append(element)
    if element == [0] * (num_regions+num_tasks):
        aligned_traces_encoded.append(aligned_trace_encoded)
        aligned_trace_encoded = []

costs = []
for gen, aln in zip(traces_generated[skip_n:], aligned_traces_encoded[skip_n:]):
    cost = edit_distance_weighted_levenshtein(gen, aln, num_regions+num_tasks, num_regions+num_tasks, hamming_distance)
    costs.append(cost)

n_eval = len(costs)
mean_cost = sum(costs) / n_eval if n_eval > 0 else float('nan')
var_cost = sum((c - mean_cost)**2 for c in costs) / n_eval if n_eval > 0 else 0.0
std_cost = var_cost ** 0.5
n_conformant = sum(1 for c in costs if c == 0)
pct_conformant = 100.0 * n_conformant / n_eval if n_eval > 0 else 0.0

print(f'Tracce totali: {len(traces_generated)}  |  valutate (post warm-up): {n_eval}')
print(f'Edit distance — media: {mean_cost:.4f}  std: {std_cost:.4f}')
print(f'Tracce conformanti: {pct_conformant:.1f}%  ({n_conformant}/{n_eval})')